# Mendelian randomization

**Purpose.** Association is not causation: BMI and coronary heart disease are
correlated, but so is almost everything with almost everything in observational
data. Mendelian randomization uses genetic variants as **instruments** to ask
whether one trait actually *causes* another. Because the genotype you carry is
fixed at conception and assigned essentially at random with respect to lifestyle,
it works like a natural randomised trial. In this notebook you will run a
two-sample MR from start to finish with the `TwoSampleMR` package, and — just as
importantly — you will test the assumptions it rests on, since MR gives a
confident-looking answer whether or not those assumptions hold.

**Learning objectives.** By the end you should be able to

- say what makes a genetic variant a valid instrument
- harmonise exposure and outcome summary statistics, and see what goes wrong if you do not
- run and compare the different MR estimators
- judge instrument strength with the F-statistic
- test for horizontal pleiotropy and heterogeneity, and read a forest, funnel and leave-one-out plot

**The data.** Two sets of **GWAS summary statistics** from the IEU OpenGWAS
database, both from studies of European ancestry:

- **exposure — body mass index**, `ieu-a-2` (Locke et al. 2015, GIANT consortium, ~340,000 individuals). Only the genome-wide significant, LD-clumped SNPs are kept, since these are the instruments.
- **outcome — coronary heart disease**, `ieu-a-7` (CARDIoGRAMplusC4D, ~185,000 individuals), restricted to those same SNPs.

No individual-level genotypes are involved — MR works entirely on summary
statistics, which is why it can combine two studies that never shared data.

## Setup

All the paths used by this exercise are set in the cell below.

In [ ]:
#############################################################
# ALL PATHS ARE SET HERE
# If the data moves, this is the ONLY cell you need to change.
# No cell below this one uses a full path.
#############################################################
DATA=/course/data/current_data/mendelian_randomization

WORK_DIR=$HOME/mendelian_randomization_human
mkdir -p $WORK_DIR
cd $WORK_DIR

# R cannot read bash variables, so write the paths to a file it can read
cat > $WORK_DIR/env.sh <<EOF
export DATA=$DATA
export WORK=$WORK_DIR
EOF

ls -l $DATA/

In [ ]:
# R cannot source env.sh, so read the paths out of it rather than repeating them
env <- readLines(path.expand("~/mendelian_randomization_human/env.sh"))
getvar <- function(k) sub(paste0('^export ', k, '='), '', grep(paste0('^export ', k, '='), env, value = TRUE)[1])
DATA <- getvar("DATA"); WORK <- getvar("WORK")
setwd(WORK)

# the two summary statistics files
ieu_a_2_rds     <- file.path(DATA, "ieu-a-2.rds")      # BMI, the exposure
ieu_a_7_out_rds <- file.path(DATA, "ieu-a-7-out.rds")  # coronary heart disease, the outcome

**Learning Objectives:**

By the end of this session, you should be able to:

- Understand what constitutes a valid genetic instrument
- Calculate and interpret F-statistics for instrument strength
- Perform MR using summary-level data
- Interpret MR estimates in the context of causality

In [ ]:
suppressWarnings(library(TwoSampleMR))
library(data.table)
library(ggplot2)

**Explanation:**

We begin by loading the required R libraries for Mendelian Randomization analysis.

- `TwoSampleMR`: the core package for performing two-sample MR
- `data.table`: for fast data manipulation
- `ggplot2`: for data visualization

### Step 1: Load BMI Exposure Data

We use BMI as our exposure. Normally, you'd extract genome-wide significant SNPs using `extract_instruments()`, but here we use a pre-downloaded dataset.

In [ ]:
# Use BMI as exposure variable
# Normally you would pull these straight from the database with
#   bmi_exp_dat <- extract_instruments(outcomes = "ieu-a-2")
# but that needs an internet connection, so we read a saved copy instead.
bmi_exp_dat <- readRDS(ieu_a_2_rds)
bmi_exp_dat

- How many SNPs are used as instruments?
- Every one of them is genome-wide significant for BMI **and** LD-clumped. Why does each of those two steps matter?

**Explanation:**

Here, we load BMI exposure data from a pre-saved RDS file. Normally, you'd use `extract_instruments()` to pull data from the MR-Base database.


**Question:**
- How many SNPs are included in this dataset?
- What are the main columns, and what do they represent (e.g., beta, se, p-values?)?

### Step 1b: Load outcome summary stats

In [ ]:
# Use coronary heart disease as outcome variable
chd_out_dat <- readRDS(ieu_a_7_out_rds)
chd_out_dat

- The outcome data contains the *same* SNPs as the exposure data, not the CHD hits. Why is it the exposure that decides which SNPs are used?

**Explanation:**

Here, we load the outcome data from a pre-saved RDS file. The SNPs have been extracted based on the selection from the exposure


**Question:**
- How many SNPs are included in this dataset and will it alway be the same as the exposure?
- Is there anything you can do if a SNP is not present in the outcome summary stats ?

### Step 2: Harmonize Exposure and Outcome Data

To ensure valid causal inference, the exposure and outcome datasets must be aligned on allele coding.

**Question:**
- Why is harmonization necessary in two-sample MR?
- What could go wrong if harmonization is skipped or misapplied?

In [ ]:
# Harmonize data
dat <- harmonise_data(bmi_exp_dat, chd_out_dat)
dat

- How many SNPs survived harmonisation, and did any get dropped?
- Harmonising means making sure the effect allele is the same allele in both studies. What would the causal estimate look like if half the SNPs were flipped?

**Explanation:**

We now harmonize the exposure and outcome datasets to ensure allele alignment. This is critical for unbiased MR estimates.

**Question:**
- What types of allele mismatches can occur?
- How does the function handle strand ambiguities?

### check hamonization

If the harmonization went well then the allele frequencies (AF) should be similar between the outcome and exposure summaries

In [ ]:
plot(dat$eaf.exposure, dat$eaf.outcome, ylab = "AF Outcome", xlab = "AF Exposure")

- The points should lie on the diagonal. What would the plot look like if the data were **not** correctly harmonised?
- Which SNPs are hardest to harmonise, and why? (Think about an A/T or C/G variant with a frequency near 0.5.)

### performing the MR


The `mr()` function performs multiple MR methods using the harmonized dataset `dat`, which contains SNP–exposure and SNP–outcome associations.



##### Methods Used

1. **Inverse-Variance Weighted (IVW)**

Assumes all SNPs are valid instruments (no pleiotropy).

$
\hat{\beta}_{\text{IVW}} = \frac{\sum_{j} w_j \cdot \hat{\beta}_{Yj} / \hat{\beta}_{Xj}}{\sum_{j} w_j}
\quad \text{where} \quad
w_j = \frac{1}{\text{SE}^2_{Yj}}
$

2. **MR-Egger Regression**

Allows for directional pleiotropy:

$
\hat{\beta}_{Yj} = \alpha + \beta_{\text{Egger}} \cdot \hat{\beta}_{Xj} + \epsilon_j
$

- \( \alpha \): intercept (tests for pleiotropy)
- \( \beta_{\text{Egger}} \): causal effect

3. **Weighted Median**

Gives a consistent estimate if >50% of SNPs are valid:

$
\hat{\beta}_{j} = \frac{\hat{\beta}_{Yj}}{\hat{\beta}_{Xj}}
$
and takes the **weighted median**.

4. **Weighted and Simple Mode**

Clusters SNPs with similar causal estimates and uses the mode (The mode of the above $\hat{\beta}_{j}$s ).


**Question:**
- What are the estimated causal effects?
- Which methods are more robust to pleiotropy?

In [ ]:
# Perform MR
res <- mr(dat)
res

### Step 3: Interpret MR Results

**Question 3:**
- What are the causal effect estimates from each MR method?
- Are they consistent?
- Which method would you trust most here, and why?

### F-statistic:
In Mendelian Randomization (MR), the **F‑statistic** assesses whether genetic variants (instruments) are **strongly associated with the exposure**.  
A common rule‑of‑thumb is:

- **F > 10** → instrument is considered *strong*  
- **F ≤ 10** → potential *weak‑instrument bias*

#### Formula for Summary‑Level (Two‑Sample) MR

For each single‑nucleotide polymorphism (SNP):

$F_j \;=\; \frac{\beta_{X,j}^{2}}{\mathrm{SE}_{\beta_{X,j}}^{2}}$

where  

- $\beta_{X,j}$ is the SNP’s estimated effect on the exposure  
- $\mathrm{SE}_{\beta_{X,j}}$ is its standard error  

For *K* SNPs, the **mean F** is

$\bar{F} \;=\; \frac{1}{K} \sum_{j=1}^{K} F_j $

In [ ]:
beta   <- bmi_exp_dat$beta.exposure
sebeta <- bmi_exp_dat$se.exposure
F      <- beta^2 / sebeta^2
mean(F)

- Is the mean F-statistic above 10, the usual rule of thumb?
- If F were below 10, what could you do about it?
- A weak instrument biases the MR estimate towards the observational association. Why is that the most dangerous direction to be biased in?

### plotting the results

In [ ]:
# Scatter plot
options(repr.plot.width = 14, repr.plot.height = 14) # make plot bigger
mr_scatter_plot(res, dat)

**Explanation:**

In the plot we visualize the MR results using scatter plot of the effect size and its SE for each SNP. 

**Question:**
- What do the visualizations tell you about the direction and consistency of the MR estimate?
- Do you se signs of heterogenerity?
- Do all of the SNPs have a significant effect on CHD?

### testing for pleiotrypy



The function `mr_pleiotropy_test(dat)` is used to detect **directional horizontal pleiotropy** in MR analyses by estimating the **intercept** of an **MR-Egger regression**.

The purpuse is to test whether SNPs affect the outcome **through pathways other than the exposure**, violating a key MR assumption.

- **Vertical pleiotropy**: SNP affects the outcome **only through the exposure** → OK for MR.
- **Horizontal pleiotropy**: SNP affects the outcome **through an alternative pathway** → Violates MR assumptions.


The MR-Egger model regresses the SNP–outcome effects on the SNP–exposure effects, allowing for an **intercept** $ \alpha $:

$
\hat{\beta}_{Yj} = \alpha + \beta_{\text{Egger}} \cdot \hat{\beta}_{Xj} + \epsilon_j
$

Where:
- $ \hat{\beta}_{Yj} $: SNP–outcome effect  
- $ \hat{\beta}_{Xj} $: SNP–exposure effect  
- $ \beta_{\text{Egger}} $: causal estimate  
- $ \alpha $: **intercept** → represents average pleiotropic effect

`mr_pleiotropy_test()` tests the null hypothesis:

$H_0: \alpha = 0$

- If $ \alpha = 0 $: No **directional pleiotropy** → MR estimates are likely unbiased.  
- If $ \alpha \ne 0 $: Evidence of **directional pleiotropy** → MR estimates may be biased.

In [ ]:
### Horizontal pleiotropy:
mr_pleiotropy_test(dat)

**Explanation:**

We now assess assumptions of the MR analysis by checking for horizontal pleiotropy .

**Question:**
- Why is horizontal pleiotropy problematic in MR?
- Do you identify horizontal pleiotropy in this data?

### testing for heterogenerity



The function `mr_heterogeneity(dat)` tests for **heterogeneity** in the Mendelian Randomization (MR) effect estimates across SNPs.  
This is similar to testing whether all SNPs provide **consistent causal estimates**, as expected under the assumption of valid instruments.



We want to evaluate whether **some SNPs have different causal effects** than others — which could indicate:

- Invalid instruments
- Horizontal pleiotropy
- Heterogeneous effects due to biology or bias

`mr_heterogeneity()` calculates **Cochran’s Q statistic**, commonly used in meta-analysis to test for heterogeneity across effect estimates.

#### For IVW:

$
Q = \sum_{j} \frac{(\hat{\beta}_{Yj} - \hat{\beta}_{Xj} \cdot \hat{\beta}_{\text{IVW}})^2}{\text{SE}_{Yj}^2}
$

Where:
- $ \hat{\beta}_{Yj} $: SNP–outcome effect  
- $ \hat{\beta}_{Xj} $: SNP–exposure effect  
- $ \hat{\beta}_{\text{IVW}} $: pooled MR estimate (from IVW)  
- $ \text{SE}_{Yj} $: standard error of SNP–outcome effect  

This is compared to a **chi-squared distribution** with $ k - 1 $ degrees of freedom (where $ k $ = number of SNPs).

In [ ]:
### Heterogeneity:
mr_heterogeneity(dat)

**Question:**
- What does a significant heterogeneity test imply?
- Why might SNPs show heterogeneous causal estimates?
- How could you proceed if you detect heterogeneity in your instruments?

### Additional robustness checks

#### effect of each snp

#### 🌲 Forest Plot: Single-SNP MR Estimates

We now generate a **forest plot** to visualize the **causal effect estimate from each SNP individually**.

In [ ]:
# Forest plot
res_single <- mr_singlesnp(dat)
mr_forest_plot(res_single)

**Question:**
- What do the visualizations tell you about the direction and consistency of the MR estimate?
- Do the individual SNPs provide consistent evidence for a causal effect?
- Are there any SNPs that appear to contradict the overall MR estimate?
- How could you follow up on SNPs that look like outliers?

####  Leave‑One‑Out (LOO) Sensitivity Analysis

We will perform a **leave‑one‑out (LOO)** analysis to check whether the overall MR result is **driven by any single SNP**.

#### What does `mr_leaveoneout()` do?

1. **Iteratively removes** one SNP at a time from the instrument set.  
2. Re‑calculates the **IVW causal estimate** on the remaining $k-1$ SNPs.  
3. Stores the new estimate $ \hat\beta_{(-j)} $ and its 95 % CI for each SNP $ j $.

In [ ]:
# Leave-one-out
res_loo <- mr_leaveoneout(dat)
mr_leaveoneout_plot(res_loo)

- **Y-axis**: SNPs that were left out  
- **X-axis**: Re-estimated causal effect $ \hat\beta_{(-j)} $  
- **Points ± CI**: The IVW estimate when that SNP is omitted  
- A **vertical reference line** marks the full IVW estimate using all SNPs


**Question:**
1. **Influential SNPs**  
   - Which SNP(s) cause the largest change in the causal estimate?  
   - What might explain their influence?

2. **Robustness**  
   - Does leaving any SNP out change the *direction* of the causal effect or only its magnitude?  
   - What does this say about the robustness of your MR conclusion?

3. **Follow-up Actions**  
   - If one SNP is clearly influential, how could you test whether it is pleiotropic?  
   - Would you exclude it, or use a pleiotropy-robust method instead?

#### Funnel Plot: Assessing Asymmetry and Pleiotropy

We generate a **funnel plot** to visually assess potential **horizontal pleiotropy** in the MR estimates from individual SNPs.

In [ ]:
# Funnel plot
res_single <- mr_singlesnp(dat)
mr_funnel_plot(res_single)

- Each point represents the **Wald ratio** estimate from a single SNP:
  $
  \hat\beta_j = \frac{\beta_{Yj}}{\beta_{Xj}}
  $
- The **X-axis** shows the causal effect estimate.
- The **Y-axis** shows the **precision** of the estimate — usually:
  $
  \text{Precision} = \frac{1}{\text{SE of } \hat\beta_j}
  $

This plot resembles a classic **funnel plot from meta-analysis**, used to detect **asymmetry**.

#### How to interpret the funnel plot

| Pattern                          | Interpretation                                         |
|----------------------------------|--------------------------------------------------------|
| Symmetric funnel                | No evidence of directional pleiotropy                 |
| Asymmetric funnel (skewed)      | Possible **directional pleiotropy** or outlier SNPs   |
| Wide spread at bottom           | Low-precision SNPs contribute noisy estimates         |

If the SNP estimates are symmetrically scattered around the pooled effect (e.g. IVW), it suggests the absence of systematic bias.



**Question:**
1. **Visual asymmetry**  
   - Do you see any skew in the plot? Are points mostly clustered on one side?

2. **Interpretation**  
   - What would an asymmetric funnel suggest about your instruments?

# Extra: the IVW estimate and the heterogeneity test by hand

The `mr()` function hides the arithmetic. It is short enough to write out, and doing so makes clear that inverse-variance weighted MR is just a weighted regression through the origin.

In [ ]:
## IVW and het test "by hand"
bx  <- dat$beta.exposure          # SNP–exposure effects  (βXj)
by  <- dat$beta.outcome           # SNP–outcome effects   (βYj)
seY <- dat$se.outcome             # SE of βYj             (SEYj)


# 2.  IVW causal estimate ----------------------------------------------------
ivw_model <- lm(by ~ bx - 1, weights = 1 / seY^2)
beta_IVW <- coef(ivw_model)
se_IVW   <- sqrt(vcov(ivw_model))

# 3.  Cochran’s Q ------------------------------------------------------------
resid <- by - bx * beta_IVW
Q <- sum((resid^2) / seY^2)

# 4.  Degrees of freedom & p‑value ------------------------------------------
df    <- length(bx) - 1
Q_p   <- pchisq(Q, df = df, lower.tail = FALSE)

# 5.  Display results --------------------------------------------------------
cat("Cochran’s Q:", round(Q, 2), "\nDF:", df,
    "\nP‑value:", signif(Q_p, 3),
    "\n beta_IVW",beta_IVW,"\n")

- Compare `beta_IVW` here with the IVW row of the `res` table above. Do they match?
- The regression has no intercept (`- 1`). What would it mean for the MR assumptions if the intercept were allowed to be non-zero and came out significantly different from zero?

### Run the cell below to take the quiz

In [ ]:
from jupyterquiz import display_quiz

display_quiz("https://raw.githubusercontent.com/popgenDK/courses/main/current_exercises/gwas/quiz/mendelian_randomization.json")